# 07 Transformer Comparison

Fine-tune MARBERT, AraBERT, or multilingual BERT and compare against the V2 TF-IDF baseline. Run this notebook on Colab with a GPU runtime.

In [16]:
from pathlib import Path
import subprocess
import sys

REPO_URL = "https://github.com/SalmaneSossey/darija-health-nlp.git"

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

# Colab starts in /content and does not have access to local WSL files.
# Clone the GitHub repo if this notebook is running outside the repository.
if not (PROJECT_ROOT / "requirements-transformers.txt").exists():
    colab_root = Path("/content/darija-health-nlp")
    if Path("/content").exists():
        if not colab_root.exists():
            print("Repository files are not present in this Colab VM. Cloning from GitHub...")
            subprocess.run(["git", "clone", REPO_URL, str(colab_root)], check=True)
        PROJECT_ROOT = colab_root
    else:
        raise FileNotFoundError("Run this notebook from the repository root or clone the GitHub repo first.")

if (PROJECT_ROOT / ".git").exists():
    print("Pulling latest code from GitHub...")
    result = subprocess.run(["git", "-C", str(PROJECT_ROOT), "pull", "--ff-only"], text=True, capture_output=True)
    print(result.stdout)
    if result.returncode != 0:
        print(result.stderr)
        print("Git pull did not fast-forward. Restart Colab runtime or delete /content/darija-health-nlp and rerun this cell.")

print("Project root:", PROJECT_ROOT)
%cd {PROJECT_ROOT}

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))


Project root: /content/darija-health-nlp
/content/darija-health-nlp


In [17]:
try:
    import torch
except ModuleNotFoundError:
    print("PyTorch is not installed in this kernel yet.")
    print("Run the dependency installation cell below, then rerun this GPU check.")
else:
    print("CUDA available:", torch.cuda.is_available())
    if torch.cuda.is_available():
        print("GPU:", torch.cuda.get_device_name(0))
    else:
        print("No GPU detected. In Colab, switch Runtime > Change runtime type > T4 GPU.")

CUDA available: True
GPU: Tesla T4


## Install optional dependencies

Run this cell in Colab or in the local `.venv` before training. The base Docker/backend setup intentionally does not include transformer dependencies.

In [18]:
import sys

requirements_path = PROJECT_ROOT / "requirements-transformers.txt"
print("Installing from:", requirements_path)
assert requirements_path.exists(), f"Missing {requirements_path}. Set PROJECT_ROOT to the repository root."
%pip install -r {requirements_path}

Installing from: /content/darija-health-nlp/requirements-transformers.txt


## Upload processed splits in Colab

From your local repo, create the zip with:

```bash
python src/data/export_colab_training_data.py
```

Then upload `artifacts/colab/darija_health_processed_splits.zip` using the next cell. Skip this upload cell if the processed CSV files already exist in Colab.


In [19]:
from pathlib import Path
from zipfile import ZipFile

processed_dir = PROJECT_ROOT / "data" / "processed"
required = [processed_dir / name for name in ["train.csv", "valid.csv", "test.csv"]]

if all(path.exists() for path in required):
    print("Processed splits already exist. Skipping upload.")
else:
    try:
        from google.colab import files
    except ImportError:
        print("Not running in Colab. Create the zip locally with src/data/export_colab_training_data.py if needed.")
    else:
        print("Upload darija_health_processed_splits.zip")
        uploaded = files.upload()
        zip_names = [name for name in uploaded if name.endswith(".zip")]
        if not zip_names:
            raise FileNotFoundError("No .zip file uploaded. Upload darija_health_processed_splits.zip.")
        zip_path = Path(zip_names[0])
        with ZipFile(zip_path) as zip_file:
            zip_file.extractall(PROJECT_ROOT)
        print("Extracted:", zip_path)
        for path in required:
            print(path, "exists=", path.exists())


Processed splits already exist. Skipping upload.


## Ensure processed splits exist

Colab cannot see your local WSL `data/` folder. Raw data, processed data, models, and artifacts are intentionally not committed to GitHub.

Before training transformers in Colab, provide data using one of these options:

1. Upload/copy `data/processed/train.csv`, `valid.csv`, and `test.csv` into this Colab repo. This is fastest.
2. Upload/copy the raw MedQA-MA dataset into `data/raw/medqa_ma`, then run the processed dataset builder.
3. Mount Google Drive and copy either the processed splits or the raw dataset from Drive.


In [20]:
from pathlib import Path

from src.utils.paths import PROCESSED_DATA_DIR, RAW_DATA_DIR

required = [PROCESSED_DATA_DIR / name for name in ["train.csv", "valid.csv", "test.csv"]]
if all(path.exists() for path in required):
    print("Processed splits found:")
    for path in required:
        print("-", path)
else:
    raw_files = list(RAW_DATA_DIR.rglob("*.csv")) if RAW_DATA_DIR.exists() else []
    if raw_files:
        print("Processed splits are missing, but raw CSV files were found. Building splits...")
        from src.data.build_processed_dataset import build_processed_dataset

        build_processed_dataset()
    else:
        missing = "\n".join(str(path) for path in required if not path.exists())
        raise FileNotFoundError(
            "Processed train/valid/test files are missing in this Colab VM.\n"
            f"Missing:\n{missing}\n\n"
            "Colab cannot access your local WSL data automatically. Upload/copy the processed CSV files "
            "to data/processed/, or upload the raw MedQA-MA dataset to data/raw/medqa_ma/ and rerun this cell."
        )


Processed splits found:
- /content/darija-health-nlp/data/processed/train.csv
- /content/darija-health-nlp/data/processed/valid.csv
- /content/darija-health-nlp/data/processed/test.csv


In [21]:
import pandas as pd

train_df = pd.read_csv(PROCESSED_DATA_DIR / "train.csv")
valid_df = pd.read_csv(PROCESSED_DATA_DIR / "valid.csv")
test_df = pd.read_csv(PROCESSED_DATA_DIR / "test.csv")
print("Train/valid/test:", train_df.shape, valid_df.shape, test_df.shape)
train_df[["text", "language", "specialty", "source"]].head()

Train/valid/test: (79714, 7) (9964, 7) (9965, 7)


,text,language,specialty,source
0,عندي مشاكل فالتحريك والتحني ديال الكتف، وخاصني...,arabic_darija,Dentistry,medqa_ma
1,صاحبتي دارت العملية سيمانة من بعد الخياطة، داب...,arabic_darija,general practitioner,medqa_ma
2,عندي 20 عام وبديت كانحس بالصداع طول الوقت، غير...,arabic_darija,Neurology,medqa_ma
3,مرحبا، بنتي عندها خمس سنين وهي كتعاني من مشكل ...,arabic_darija,Pediatric Medicine,medqa_ma
4,واخا انا مترجم محترف ومتخصص فتحويل النصوص من ا...,arabic_darija,Rheumatology and Orthopedics,medqa_ma


## Verify transformer training code

This cell confirms that Colab is using the current GitHub version of the training script, not a stale imported module.


In [ ]:
from pathlib import Path

script_path = PROJECT_ROOT / "src" / "models" / "train_transformer_specialty_classifier.py"
script_text = script_path.read_text()
uses_compat_wrapper = "trainer_kwargs" in script_text and "Trainer(**trainer_kwargs)" in script_text
supports_new_api = "processing_class" in script_text
print("Training script:", script_path)
print("Uses Trainer compatibility wrapper:", uses_compat_wrapper)
print("Supports new Transformers processing_class API:", supports_new_api)
assert uses_compat_wrapper, "Stale training script detected. Rerun the setup cell, or restart the Colab runtime and open the latest notebook."


## Train one transformer

Start with MARBERT. It is a reasonable first choice for Arabic dialectal text. Keep TF-IDF + LinearSVC as the baseline and compare against its test macro F1 of about `0.6255`.

In [22]:
import importlib
import src.models.train_transformer_specialty_classifier as transformer_training

# Rerunning cells in Colab can keep old imported code in memory.
# Reload so the latest `git pull` changes are used.
transformer_training = importlib.reload(transformer_training)
MODEL_CHOICES = transformer_training.MODEL_CHOICES
train_transformer = transformer_training.train_transformer

MODEL_CHOICE = "marbert"  # options: marbert, arabert, mbert, or any Hugging Face model id
LABEL_MODE = "specialty"  # options: specialty, broad, rare_merged
EPOCHS = 3
BATCH_SIZE = 8
MAX_LENGTH = 160

model_id = MODEL_CHOICES.get(MODEL_CHOICE, MODEL_CHOICE)
print("Training:", model_id)

train_transformer(
    model_name=model_id,
    label_mode=LABEL_MODE,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    max_length=MAX_LENGTH,
)


Training: UBC-NLP/MARBERT


Map:   0%|          | 0/79714 [00:00<?, ? examples/s]

Map:   0%|          | 0/9964 [00:00<?, ? examples/s]

Map:   0%|          | 0/9965 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: UBC-NLP/MARBERT
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on yo

TypeError: Trainer.__init__() got an unexpected keyword argument 'tokenizer'

## Inspect transformer metrics

In [ ]:
import json
from pathlib import Path

from src.utils.paths import METRICS_DIR

metric_files = sorted(METRICS_DIR.glob("transformer_*_metrics.json"))
for path in metric_files:
    metrics = json.loads(path.read_text())
    print("\n", path.name)
    for key, value in metrics.items():
        if isinstance(value, float):
            print(f"{key}: {value:.4f}")
        else:
            print(f"{key}: {value}")

## Compare against V2 classical baseline

In [ ]:
baseline_path = METRICS_DIR / "v2_specialty_metrics.json"
if baseline_path.exists():
    baseline = json.loads(baseline_path.read_text())
    print("V2 char_wb TF-IDF LinearSVC test metrics")
    print("accuracy:", round(baseline["accuracy"], 4))
    print("macro_f1:", round(baseline["macro_f1"], 4))
    print("weighted_f1:", round(baseline["weighted_f1"], 4))
else:
    print("Run src/models/analyze_v2_errors.py to generate v2_specialty_metrics.json")